In [1]:
import sys

if len(sys.argv) != 3:
    print('Please give two seq-files.')
    print('Necessary using the FASTA format.')
    exit()


匯入sys套件，使用其環境變數"argv"的功能
判斷使用者是否提供兩個FASTA格式的序列檔案

In [2]:
seq1 = ' '
with open(sys.argv[1], 'r') as f:
    for line in f:
        if line[0] != '>':
            seq1 += line.strip()

seq2 = ' '
with open(sys.argv[2], 'r') as f:
    for line in f:
        if line[0] != '>':
            seq2 += line.strip()




FileNotFoundError: [Errno 2] No such file or directory: '-f'

使用with open()函式一次讀取一行資料。    
FASTA格式會在第一行以">"為開頭作為註解，因此不考慮讀取該行。

In [3]:
def scoringAndPathing(matrix, i, j):

    if i == j == 0:
        return 0, -1
    elif i == 0:
        return matrix[i][j-1] - 2, 0
    elif j == 0:
        return matrix[i-1][j] - 2, 1
    else:
        match_score = 1 if seq1[i] == seq2[j] else -1
        tmp = [matrix[i][j-1] - 2, matrix[i-1][j] - 2, matrix[i-1][j-1] + match_score]
        path = max(range(len(tmp)), key=lambda i:tmp[i])
        return tmp[path], path



定義評分方式：    
match = +1    
missmatch = -1    
gap = -2    
計算分數與路徑矩陣的函數

In [4]:
def score_print(matrix, seq1, seq2):
    print('The similarity array is listed below:')
    print(" ", end='')
    for i in seq2:
        print('{:>5s}'.format(i), end='')
    print('')
    for i in range(len(seq1)):
        print(seq1[i], end='')
        for j in matrix[i]:
            print('{:>5d}'.format(j), end='')
        print('')

    print('')


定義格式化輸出分數矩陣。

In [5]:
def trace_back(matrix, seq1, seq2):
    seq1_result = ''
    seq2_result = ''
    match_result = ''
    i, j = len(seq1) - 1, len(seq2) - 1
    while True:
        if matrix[i][j] == 2:
            seq1_result = seq1[i] + seq1_result
            seq2_result = seq2[j] + seq2_result
            tmp_match = '|' if seq1[i] == seq2[j] else ' '
            match_result = tmp_match + match_result
            i, j = i - 1, j - 1
        elif matrix[i][j] == 1:
            seq1_result = seq1[i] + seq1_result
            seq2_result = '-' + seq2_result
            match_result = ' ' + match_result
            i -= 1
        elif matrix[i][j] == 0:
            seq1_result = '-' + seq1_result
            seq2_result = seq2[j] + seq2_result
            match_result = ' ' + match_result
            j -= 1
        elif matrix[i][j] == -1:
            break

    return seq1_result, seq2_result, match_result

定義追溯函式，回傳含有gap的序列以及比對結果。

In [6]:
def align_print(seq1, seq2, match):
    print('===================================================')
    print('Globally optimal alignment with dynamic programming')
    print('===================================================')
    print('aligned len =', len(seq1))
    num_lines = len(seq1) // 60
    remainder = len(seq1) % 60
    for i in range(num_lines + 1):
        print()
        if i != num_lines:
            out_str1 = seq1[i*60:i*60+60]
            out_str2 = seq2[i*60:i*60+60]
            out_str3 = match[i*60:i*60+60]
            shift = 60
        elif remainder != 0:
            out_str1 = seq1[i*60:]
            out_str2 = seq2[i*60:]
            out_str3 = match[i*60:]
            shift = remainder
        else:
            break

        print('Sbjct:','{:>5d}'.format(i*60+1),out_str1,'{:<5d}'.format(i*60+shift))
        print('      ','{:>5s}'.format(''),out_str3,'{:<5s}'.format(''))
        print('Query:','{:>5d}'.format(i*60+1),out_str2,'{:<5d}'.format(i*60+shift))


In [8]:
seq1 = " ATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGAT"
seq2 = " AGCTTTCGA"

score_matrix = [[0 for j in range(len(seq2))] for i in range(len(seq1))]
path_matrix = [[0 for j in range(len(seq2))] for i in range(len(seq1))]

for i in range(len(seq1)):
    for j in range(len(seq2)):
        score_matrix[i][j], path_matrix[i][j] = scoringAndPathing(score_matrix, i, j)

In [9]:
score_print(score_matrix, seq1, seq2)

The similarity array is listed below:
          A    G    C    T    T    T    C    G    A
     0   -2   -4   -6   -8  -10  -12  -14  -16  -18
A   -2    1   -1   -3   -5   -7   -9  -11  -13  -15
T   -4   -1    0   -2   -2   -4   -6   -8  -10  -12
G   -6   -3    0   -1   -3   -3   -5   -7   -7   -9
C   -8   -5   -2    1   -1   -3   -4   -4   -6   -8
G  -10   -7   -4   -1    0   -2   -4   -5   -3   -5
C  -12   -9   -6   -3   -2   -1   -3   -3   -5   -4
T  -14  -11   -8   -5   -2   -1    0   -2   -4   -6
C  -16  -13  -10   -7   -4   -3   -2    1   -1   -3
G  -18  -15  -12   -9   -6   -5   -4   -1    2    0
A  -20  -17  -14  -11   -8   -7   -6   -3    0    3
A  -22  -19  -16  -13  -10   -9   -8   -5   -2    1
T  -24  -21  -18  -15  -12   -9   -8   -7   -4   -1
G  -26  -23  -20  -17  -14  -11  -10   -9   -6   -3
C  -28  -25  -22  -19  -16  -13  -12   -9   -8   -5
G  -30  -27  -24  -21  -18  -15  -14  -11   -8   -7
C  -32  -29  -26  -23  -20  -17  -16  -13  -10   -9
T  -34  -31  -28  -25  -22

In [10]:
result1, result2, result3 = trace_back(path_matrix, seq1, seq2)

In [11]:
align_print(result1, result2, result3)

Globally optimal alignment with dynamic programming
aligned len = 61

Sbjct:     1 ATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGAATGCGCTCGA 60   
             | ||  |    |    ||||                                              
Query:     1 A-GC--T----T----TCGA---------------------------------------- 60   

Sbjct:    61 T 61   
                    
Query:    61 - 61   
